In [1]:
import os
import re
import numpy as np
import pandas as pd

from pandas.errors import EmptyDataError
from scipy.signal import butter, filtfilt, find_peaks, welch
from scipy.interpolate import interp1d


# =============================================================================
# CONFIG
# =============================================================================
DATA_ROOT = r"/content/drive/MyDrive/Colab Notebooks/datasets/Wearable-device-dataset/Wearable_Dataset"
STRESS_ROOT = os.path.join(DATA_ROOT, "STRESS")

OUTPUT_DIR = r"/content/drive/MyDrive/Colab Notebooks/BM/TPV-stress/Wearable-Device-Dataset/HRV"
os.makedirs(OUTPUT_DIR, exist_ok=True)

FS_BVP = 64

WINDOW_LIST_SEC = [60, 120]
STEP_RATIO = 0.5

STRESS_RATIO_THR = 0.50
REST_RATIO_THR = 0.10
USE_DISCARD = True

LOWCUT = 0.5
HIGHCUT = 8.0
FILTER_ORDER = 4

MIN_HR_BPM = 40
MAX_HR_BPM = 180
MIN_PEAK_DISTANCE_SEC = 60.0 / MAX_HR_BPM

IBI_MIN_SEC = 0.30
IBI_MAX_SEC = 1.50
IBI_DIFF_RATIO_MAX = 0.30

FS_IBI_INTERP = 4.0
LF_LOW = 0.04
LF_HIGH = 0.15
HF_LOW = 0.15
HF_HIGH = 0.40

USE_SEGMENTS = {
    "First Rest",
    "Second Rest",
    "TMCT",
    "Real Opinion",
    "Opposite Opinion",
    "Subtract Test",
}


# =============================================================================
# Loaders
# =============================================================================
def load_empatica_1d_csv(file_path, signal_name):
    raw = pd.read_csv(file_path, header=None)

    start_time = pd.to_datetime(raw.iloc[0, 0])
    fs = float(raw.iloc[1, 0])

    values = raw.iloc[2:, 0].astype(float).reset_index(drop=True).values
    timestamps = start_time + pd.to_timedelta(np.arange(len(values)) / fs, unit="s")

    df = pd.DataFrame({
        "time": timestamps,
        signal_name: values
    })

    return df, start_time, fs


def load_tags_csv(file_path):
    if os.path.getsize(file_path) == 0:
        return []

    try:
        tags = pd.read_csv(file_path, header=None)
    except EmptyDataError:
        return []

    if tags.empty:
        return []

    tags[0] = pd.to_datetime(tags[0])
    return tags[0].tolist()


# =============================================================================
# v2 protocol intervals
# =============================================================================
def build_v2_stress_intervals(tags):
    if len(tags) < 9:
        raise ValueError(f"v2 requires at least 9 tags, got {len(tags)}")

    specs = [
        ("TMCT",             tags[1], tags[2], 1),
        ("First Rest",       tags[2], tags[3], 0),
        ("Real Opinion",     tags[3], tags[4], 1),
        ("Opposite Opinion", tags[5], tags[6], 1),
        ("Second Rest",      tags[6], tags[7], 0),
        ("Subtract Test",    tags[7], tags[8], 1),
    ]

    rows = []
    for idx, (phase, start, end, label) in enumerate(specs):
        if phase not in USE_SEGMENTS:
            continue
        if end <= start:
            continue

        rows.append({
            "interval_idx": idx,
            "phase": phase,
            "start": start,
            "end": end,
            "label": int(label),
            "status_name": "stress" if label == 1 else "normal",
            "duration_sec": (end - start).total_seconds()
        })

    return pd.DataFrame(rows)


def overlap_sec(a_start, a_end, b_start, b_end):
    start = max(a_start, b_start)
    end = min(a_end, b_end)
    sec = (end - start).total_seconds()
    return max(0.0, sec)


def build_window_table_from_intervals(intervals_df, window_sec=60, step_sec=30):
    rows = []

    timeline_start = intervals_df["start"].min()
    timeline_end = intervals_df["end"].max()

    win_delta = pd.to_timedelta(window_sec, unit="s")
    step_delta = pd.to_timedelta(step_sec, unit="s")

    t0 = timeline_start
    window_idx = 0

    while t0 + win_delta <= timeline_end:
        t1 = t0 + win_delta
        window_idx += 1

        stress_overlap = 0.0
        rest_overlap = 0.0
        phase_overlaps = {}

        for _, r in intervals_df.iterrows():
            ov = overlap_sec(t0, t1, r["start"], r["end"])

            if ov <= 0:
                continue

            phase_overlaps[r["phase"]] = ov

            if int(r["label"]) == 1:
                stress_overlap += ov
            else:
                rest_overlap += ov

        stress_ratio = stress_overlap / window_sec
        rest_ratio = rest_overlap / window_sec
        covered_ratio = (stress_overlap + rest_overlap) / window_sec

        if USE_DISCARD:
            if stress_ratio >= STRESS_RATIO_THR:
                label = 1
                status_name = "stress"
            elif stress_ratio <= REST_RATIO_THR and rest_ratio > 0:
                label = 0
                status_name = "normal"
            else:
                t0 += step_delta
                continue
        else:
            if stress_overlap >= rest_overlap:
                label = 1
                status_name = "stress"
            else:
                label = 0
                status_name = "normal"

        dominant_phase = max(phase_overlaps, key=phase_overlaps.get) if phase_overlaps else "none"

        rows.append({
            "phase": dominant_phase,
            "label": label,
            "status_name": status_name,
            "window_start": t0,
            "window_end": t1,
            "major_ratio": max(stress_ratio, rest_ratio),
            "stress_ratio": stress_ratio,
            "rest_ratio": rest_ratio,
            "covered_ratio": covered_ratio,
        })

        t0 += step_delta

    return pd.DataFrame(rows)


# =============================================================================
# Basic utils
# =============================================================================
def fill_nan_with_median(x):
    x = np.asarray(x, dtype=np.float32).reshape(-1).copy()
    if np.isnan(x).any():
        med = np.nanmedian(x)
        if np.isnan(med):
            med = 0.0
        x = np.nan_to_num(x, nan=med, posinf=med, neginf=med)
    return x


def bandpass_filter(sig, fs=64, low=0.5, high=8.0, order=4):
    sig = np.asarray(sig, dtype=np.float32).reshape(-1)
    nyq = 0.5 * fs
    low_n = low / nyq
    high_n = high / nyq

    if not (0 < low_n < high_n < 1):
        raise ValueError(f"Invalid bandpass range: low={low}, high={high}, fs={fs}")

    b, a = butter(order, [low_n, high_n], btype="band")
    return filtfilt(b, a, sig).astype(np.float32)


def zscore_1d(x):
    x = np.asarray(x, dtype=np.float32).reshape(-1)
    mu = float(np.mean(x))
    sd = float(np.std(x))
    if sd < 1e-8:
        return np.zeros_like(x, dtype=np.float32)
    return ((x - mu) / (sd + 1e-8)).astype(np.float32)


def safe_div(a, b):
    return float(a / b) if abs(b) > 1e-12 else np.nan


# =============================================================================
# HRV helpers
# =============================================================================
def detect_ppg_peaks(sig, fs):
    sig = np.asarray(sig, dtype=np.float32).reshape(-1)
    distance = max(1, int(round(MIN_PEAK_DISTANCE_SEC * fs)))
    prom = max(0.10, 0.20 * float(np.std(sig)))
    peaks, props = find_peaks(sig, distance=distance, prominence=prom)
    return peaks, props


def compute_basic_peak_info(peaks, fs):
    peaks = np.asarray(peaks, dtype=np.int64)

    if len(peaks) < 2:
        return {
            "n_peaks": int(len(peaks)),
            "ibi_mean_sec_raw": np.nan,
            "ibi_pass_ratio": 0.0,
            "valid_ibi_count": 0,
        }

    ibi_raw = np.diff(peaks) / float(fs)
    plausible = (ibi_raw >= IBI_MIN_SEC) & (ibi_raw <= IBI_MAX_SEC)
    ibi_pass_ratio = float(np.mean(plausible)) if len(ibi_raw) > 0 else 0.0

    return {
        "n_peaks": int(len(peaks)),
        "ibi_mean_sec_raw": float(np.mean(ibi_raw)) if len(ibi_raw) > 0 else np.nan,
        "ibi_pass_ratio": ibi_pass_ratio,
        "valid_ibi_count": int(np.sum(plausible)),
    }


def build_filtered_ibi_series(peaks, fs):
    peaks = np.asarray(peaks, dtype=np.int64)

    if len(peaks) < 3:
        return None

    peak_times = peaks.astype(np.float64) / float(fs)
    ibi = np.diff(peak_times)
    ibi_times = peak_times[1:]

    if len(ibi) < 2:
        return None

    plausible = (ibi >= IBI_MIN_SEC) & (ibi <= IBI_MAX_SEC)
    ibi = ibi[plausible]
    ibi_times = ibi_times[plausible]

    if len(ibi) < 2:
        return None

    med_ibi = float(np.median(ibi))
    stable = np.abs(ibi - med_ibi) <= (IBI_DIFF_RATIO_MAX * (med_ibi + 1e-8))

    ibi = ibi[stable]
    ibi_times = ibi_times[stable]

    if len(ibi) < 2:
        return None

    return {
        "peak_times": peak_times,
        "ibi_times": ibi_times,
        "ibi": ibi,
        "ibi_median_sec": med_ibi,
    }


def interpolate_ibi(ibi_times, ibi, fs_interp=FS_IBI_INTERP):
    ibi_times = np.asarray(ibi_times, dtype=np.float64).reshape(-1)
    ibi = np.asarray(ibi, dtype=np.float64).reshape(-1)

    if len(ibi_times) < 2 or len(ibi) < 2:
        return None

    if ibi_times[-1] <= ibi_times[0]:
        return None

    t_uniform = np.arange(ibi_times[0], ibi_times[-1], 1.0 / fs_interp)

    if len(t_uniform) < 4:
        return None

    try:
        f_interp = interp1d(
            ibi_times,
            ibi,
            kind="linear",
            bounds_error=False,
            fill_value="extrapolate",
            assume_sorted=True
        )
        ibi_uniform = f_interp(t_uniform)
    except Exception:
        return None

    ibi_uniform = np.asarray(ibi_uniform, dtype=np.float64)

    if np.any(~np.isfinite(ibi_uniform)):
        return None

    return {
        "t_uniform": t_uniform,
        "ibi_uniform": ibi_uniform,
    }


def compute_freq_domain_hrv(ibi_uniform, fs_interp=FS_IBI_INTERP):
    ibi_uniform = np.asarray(ibi_uniform, dtype=np.float64).reshape(-1)

    if len(ibi_uniform) < 8:
        return {
            "LF": np.nan,
            "HF": np.nan,
            "LF_HF": np.nan,
        }

    nperseg = min(256, len(ibi_uniform))
    f, pxx = welch(ibi_uniform, fs=fs_interp, nperseg=nperseg)

    lf_mask = (f >= LF_LOW) & (f < LF_HIGH)
    hf_mask = (f >= HF_LOW) & (f < HF_HIGH)

    lf = np.trapz(pxx[lf_mask], f[lf_mask]) if np.any(lf_mask) else np.nan
    hf = np.trapz(pxx[hf_mask], f[hf_mask]) if np.any(hf_mask) else np.nan
    lf_hf = safe_div(lf, hf) if np.isfinite(lf) and np.isfinite(hf) else np.nan

    return {
        "LF": float(lf) if np.isfinite(lf) else np.nan,
        "HF": float(hf) if np.isfinite(hf) else np.nan,
        "LF_HF": float(lf_hf) if np.isfinite(lf_hf) else np.nan,
    }


def compute_time_domain_hrv(ibi):
    ibi = np.asarray(ibi, dtype=np.float64).reshape(-1)

    if len(ibi) < 2:
        return {
            "RMSSD": np.nan,
            "SDNN": np.nan,
            "ibi_mean_sec": np.nan,
            "ibi_std_sec": np.nan,
        }

    diff_ibi = np.diff(ibi)

    rmssd = np.sqrt(np.mean(diff_ibi ** 2)) if len(diff_ibi) > 0 else np.nan
    sdnn = np.std(ibi, ddof=0)

    return {
        "RMSSD": float(rmssd) if np.isfinite(rmssd) else np.nan,
        "SDNN": float(sdnn) if np.isfinite(sdnn) else np.nan,
        "ibi_mean_sec": float(np.mean(ibi)) if np.isfinite(np.mean(ibi)) else np.nan,
        "ibi_std_sec": float(np.std(ibi, ddof=0)) if np.isfinite(np.std(ibi, ddof=0)) else np.nan,
    }


def extract_hrv_features_from_bvp_segment(seg_bvp_raw, fs_bvp):
    seg_bvp = fill_nan_with_median(seg_bvp_raw)
    seg_bvp_bp = bandpass_filter(seg_bvp, fs=fs_bvp, low=LOWCUT, high=HIGHCUT, order=FILTER_ORDER)
    seg_bvp_z = zscore_1d(seg_bvp_bp)

    peaks, props = detect_ppg_peaks(seg_bvp_z, fs=fs_bvp)
    peak_info = compute_basic_peak_info(peaks, fs_bvp)

    ibi_pack = build_filtered_ibi_series(peaks, fs_bvp)

    row = {
        "n_peaks": peak_info["n_peaks"],
        "ibi_pass_ratio": peak_info["ibi_pass_ratio"],
        "valid_ibi_count": 0,
        "ibi_mean_sec_raw": peak_info["ibi_mean_sec_raw"],
        "ibi_mean_sec": np.nan,
        "ibi_std_sec": np.nan,
        "RMSSD": np.nan,
        "SDNN": np.nan,
        "LF": np.nan,
        "HF": np.nan,
        "LF_HF": np.nan,
    }

    if ibi_pack is None:
        return row

    ibi = ibi_pack["ibi"]
    ibi_times = ibi_pack["ibi_times"]

    row["valid_ibi_count"] = int(len(ibi))

    td = compute_time_domain_hrv(ibi)
    row.update(td)

    interp_pack = interpolate_ibi(ibi_times, ibi, fs_interp=FS_IBI_INTERP)

    if interp_pack is None:
        return row

    fd = compute_freq_domain_hrv(interp_pack["ibi_uniform"], fs_interp=FS_IBI_INTERP)
    row.update(fd)

    return row


# =============================================================================
# Subject HRV extraction
# =============================================================================
def build_subject_hrv_table(subject_dir, window_sec):
    subject_id = os.path.basename(subject_dir)

    bvp_path = os.path.join(subject_dir, "BVP.csv")
    tags_path = os.path.join(subject_dir, "tags.csv")

    if not (os.path.exists(bvp_path) and os.path.exists(tags_path)):
        print(f"[WARN] {subject_id}: missing BVP.csv or tags.csv")
        return pd.DataFrame()

    tags = load_tags_csv(tags_path)

    if len(tags) < 9:
        print(f"[WARN] {subject_id}: invalid tags count={len(tags)}")
        return pd.DataFrame()

    bvp_df, _, fs_bvp = load_empatica_1d_csv(bvp_path, "bvp")

    if int(round(fs_bvp)) != FS_BVP:
        print(f"[WARN] {subject_id}: BVP fs mismatch. Expected={FS_BVP}, got={fs_bvp}")

    intervals_df = build_v2_stress_intervals(tags)

    step_sec = int(window_sec * STEP_RATIO)

    window_df = build_window_table_from_intervals(
        intervals_df,
        window_sec=window_sec,
        step_sec=step_sec
    )

    rows = []
    window_counter = 0

    expected_bvp_len = int(window_sec * FS_BVP)

    for _, w in window_df.iterrows():
        t0 = w["window_start"]
        t1 = w["window_end"]

        seg_bvp_raw = bvp_df[
            (bvp_df["time"] >= t0) &
            (bvp_df["time"] < t1)
        ]["bvp"].values

        if len(seg_bvp_raw) != expected_bvp_len:
            continue

        hrv_info = extract_hrv_features_from_bvp_segment(seg_bvp_raw, FS_BVP)

        window_counter += 1

        row = {
            "subject": subject_id,
            "window": f"window{window_counter}",
            "window_index": window_counter,
            "window_sec": int(window_sec),
            "status": int(w["label"]),
            "status_name": "stress" if int(w["label"]) == 1 else "normal",
            "phase": w["phase"],
            "t_start": t0,
            "t_end": t1,
            "major_ratio": float(w["major_ratio"]),
            "stress_ratio": float(w["stress_ratio"]),
            "rest_ratio": float(w["rest_ratio"]),
            "covered_ratio": float(w["covered_ratio"]),
        }

        row.update(hrv_info)
        rows.append(row)

    return pd.DataFrame(rows)


def build_all_subjects_hrv_table(window_sec):
    subject_dirs = sorted([
        os.path.join(STRESS_ROOT, d)
        for d in os.listdir(STRESS_ROOT)
        if re.fullmatch(r"f\d{2}", d)
        and os.path.isdir(os.path.join(STRESS_ROOT, d))
    ])

    all_dfs = []

    print(f"[INFO] Found v2 subject folders: {len(subject_dirs)}")
    print([os.path.basename(d) for d in subject_dirs])
    print(f"[INFO] Building HRV table for window_sec={window_sec}, step_sec={int(window_sec * STEP_RATIO)}")

    for subject_dir in subject_dirs:
        sid = os.path.basename(subject_dir)
        print(f"\n[INFO] Processing {sid} ...")

        try:
            df_sub = build_subject_hrv_table(subject_dir, window_sec=window_sec)
            print(f"[INFO] {sid}: extracted {len(df_sub)} windows")

            if len(df_sub) > 0:
                print(df_sub["status_name"].value_counts().to_dict())
                print(df_sub["phase"].value_counts().to_dict())
                all_dfs.append(df_sub)

        except Exception as e:
            print(f"[WARN] Failed on {sid}: {e}")

    if len(all_dfs) == 0:
        return pd.DataFrame()

    return pd.concat(all_dfs, axis=0, ignore_index=True)


# =============================================================================
# Save
# =============================================================================
def save_hrv_outputs(df_all, window_sec, output_dir):
    if len(df_all) == 0:
        print(f"[WARN] No windows extracted for {window_sec}s")
        return

    base_cols = [
        "subject", "window", "window_index", "window_sec",
        "status", "status_name", "phase",
        "t_start", "t_end",
        "major_ratio", "stress_ratio", "rest_ratio", "covered_ratio"
    ]

    hrv_cols = [
        "n_peaks",
        "ibi_pass_ratio",
        "valid_ibi_count",
        "ibi_mean_sec_raw",
        "ibi_mean_sec",
        "ibi_std_sec",
        "RMSSD",
        "SDNN",
        "LF",
        "HF",
        "LF_HF",
    ]

    final_cols = base_cols + hrv_cols
    df_all = df_all[final_cols].copy()

    minute_tag = f"{window_sec // 60}min"

    mode_tag = "discard" if USE_DISCARD else "no_discard"

    csv_path = os.path.join(output_dir, f"Wearable_v2_HRV_{minute_tag}_{mode_tag}.csv")
    df_all.to_csv(csv_path, index=False, encoding="utf-8-sig")
    print(f"[INFO] Saved: {csv_path}")

    summary = (
        df_all.groupby(["subject", "status_name"])
        .agg(
            n_windows=("window", "count"),
            n_valid_rmssd=("RMSSD", lambda x: int(np.sum(np.isfinite(x)))),
            n_valid_lf_hf=("LF_HF", lambda x: int(np.sum(np.isfinite(x)))),
        )
        .reset_index()
    )

    summary_csv = os.path.join(output_dir, f"Wearable_v2_HRV_{minute_tag}_{mode_tag}_summary.csv")
    summary.to_csv(summary_csv, index=False, encoding="utf-8-sig")
    print(f"[INFO] Saved: {summary_csv}")

    phase_summary = (
        df_all.groupby(["subject", "phase", "status_name"])
        .agg(
            n_windows=("window", "count"),
            n_valid_rmssd=("RMSSD", lambda x: int(np.sum(np.isfinite(x)))),
            n_valid_lf_hf=("LF_HF", lambda x: int(np.sum(np.isfinite(x)))),
        )
        .reset_index()
    )

    phase_summary_csv = os.path.join(output_dir, f"Wearable_v2_HRV_{minute_tag}_{mode_tag}_phase_summary.csv")
    phase_summary.to_csv(phase_summary_csv, index=False, encoding="utf-8-sig")
    print(f"[INFO] Saved: {phase_summary_csv}")

    print(f"\n[INFO] Status counts ({minute_tag})")
    print(df_all["status_name"].value_counts())

    print(f"\n[INFO] Phase counts ({minute_tag})")
    print(df_all["phase"].value_counts())

    print(f"\n[INFO] Summary ({minute_tag})")
    print(summary.to_string(index=False))


# =============================================================================
# Main
# =============================================================================
if __name__ == "__main__":
    for window_sec in WINDOW_LIST_SEC:
        df_all = build_all_subjects_hrv_table(window_sec=window_sec)

        print(f"\n[INFO] final raw shape for {window_sec}s:", df_all.shape)

        if len(df_all) == 0:
            print(f"[WARN] No target windows extracted for {window_sec}s")
            continue

        save_hrv_outputs(df_all, window_sec=window_sec, output_dir=OUTPUT_DIR)

[INFO] Found v2 subject folders: 17
['f01', 'f02', 'f03', 'f04', 'f05', 'f06', 'f07', 'f08', 'f09', 'f10', 'f11', 'f12', 'f13', 'f15', 'f16', 'f17', 'f18']
[INFO] Building HRV table for window_sec=60, step_sec=30

[INFO] Processing f01 ...


/tmp/ipykernel_11663/3125235093.py:375: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  lf = np.trapz(pxx[lf_mask], f[lf_mask]) if np.any(lf_mask) else np.nan
/tmp/ipykernel_11663/3125235093.py:376: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  hf = np.trapz(pxx[hf_mask], f[hf_mask]) if np.any(hf_mask) else np.nan


[INFO] f01: extracted 82 windows
{'normal': 64, 'stress': 18}
{'Second Rest': 37, 'First Rest': 27, 'TMCT': 15, 'Real Opinion': 2, 'Opposite Opinion': 1}

[INFO] Processing f02 ...


/tmp/ipykernel_11663/3125235093.py:375: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  lf = np.trapz(pxx[lf_mask], f[lf_mask]) if np.any(lf_mask) else np.nan
/tmp/ipykernel_11663/3125235093.py:376: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  hf = np.trapz(pxx[hf_mask], f[hf_mask]) if np.any(hf_mask) else np.nan


[INFO] f02: extracted 52 windows
{'normal': 38, 'stress': 14}
{'First Rest': 25, 'Second Rest': 13, 'TMCT': 12, 'Real Opinion': 1, 'Opposite Opinion': 1}

[INFO] Processing f03 ...


...

[INFO] f16: extracted 27 windows
{'normal': 23, 'stress': 4}
{'First Rest': 14, 'Second Rest': 9, 'TMCT': 4}

[INFO] Processing f17 ...


/tmp/ipykernel_11663/3125235093.py:375: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  lf = np.trapz(pxx[lf_mask], f[lf_mask]) if np.any(lf_mask) else np.nan
/tmp/ipykernel_11663/3125235093.py:376: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  hf = np.trapz(pxx[hf_mask], f[hf_mask]) if np.any(hf_mask) else np.nan


[INFO] f17: extracted 29 windows
{'normal': 24, 'stress': 5}
{'First Rest': 12, 'Second Rest': 12, 'TMCT': 4, 'Real Opinion': 1}

[INFO] Processing f18 ...


/tmp/ipykernel_11663/3125235093.py:375: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  lf = np.trapz(pxx[lf_mask], f[lf_mask]) if np.any(lf_mask) else np.nan
/tmp/ipykernel_11663/3125235093.py:376: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  hf = np.trapz(pxx[hf_mask], f[hf_mask]) if np.any(hf_mask) else np.nan


[INFO] f18: extracted 25 windows
{'normal': 21, 'stress': 4}
{'First Rest': 12, 'Second Rest': 9, 'TMCT': 3, 'Opposite Opinion': 1}

[INFO] final raw shape for 120s: (499, 24)
[INFO] Saved: /content/drive/MyDrive/Colab Notebooks/BM/TPV-stress/Wearable-Device-Dataset/HRV/Wearable_v2_HRV_2min_discard.csv
[INFO] Saved: /content/drive/MyDrive/Colab Notebooks/BM/TPV-stress/Wearable-Device-Dataset/HRV/Wearable_v2_HRV_2min_discard_summary.csv
[INFO] Saved: /content/drive/MyDrive/Colab Notebooks/BM/TPV-stress/Wearable-Device-Dataset/HRV/Wearable_v2_HRV_2min_discard_phase_summary.csv

[INFO] Status counts (2min)
status_name
normal    414
stress     85
Name: count, dtype: int64

[INFO] Phase counts (2min)
phase
First Rest          220
Second Rest         194
TMCT                 71
Opposite Opinion      9
Real Opinion          5
Name: count, dtype: int64

[INFO] Summary (2min)
subject status_name  n_windows  n_valid_rmssd  n_valid_lf_hf
    f01      normal         29             29             2